# **Language Model and Application for Spelling Error Correction**

## Import libraries

In [40]:
import nltk
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
from collections import Counter, defaultdict
import math
import string

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download("stopwords")
nltk.download("wordnet")

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


## Import dataset

In [41]:
from google.colab import drive
drive.mount('/content/drive')
txt_path = '/content/drive/My Drive/Colab Notebooks/Data files/tedtalk.txt'


with open(txt_path, "r", encoding="utf-8") as f:
    data = f.read()

# Print data for testing
print(data[:100])


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Thank you so much, Chris. And it's truly a great honor to have the opportunity to come to this stage


## Data preprocessing

In [42]:
def clean_text(text):
  if text is None:
    return ""
  text = re.sub(r"[^a-zA-Z0-9\s.,!?]", "", text)
  text = text.strip()
  return text

def remove_stopwords(tokens):
  if tokens is None:
    return []
  stop_words = set(stopwords.words("english"))
  return [word for word in tokens if word not in stop_words]

def lemmatize(tokens):
  lemmatizer = WordNetLemmatizer()
  return [lemmatizer.lemmatize(word) for word in tokens]

In [43]:
def preprocessing(text):
  if text is None:
    return []
  # Clean text
  cleaned_text = clean_text(text)
  # Lower text to lowercase
  cleaned_text = cleaned_text.lower()
  # Remove punctuation
  cleaned_text = cleaned_text.translate(str.maketrans("", "", string.punctuation))
  # Tokenize text
  tokens = nltk.word_tokenize(cleaned_text)
  # Remove stopwords
  tokens = remove_stopwords(tokens)
  # Lemmatize
  tokens = lemmatize(tokens)
  final_tokens = [token for token in tokens if token.strip()]
  return final_tokens

### Preprocessing to tokens list (words).

In [44]:
tokens = preprocessing(data)
print("Number of tokens:", len(tokens))
print(tokens[:100])

Number of tokens: 3527657
['thank', 'much', 'chris', 'truly', 'great', 'honor', 'opportunity', 'come', 'stage', 'twice', 'im', 'extremely', 'grateful', 'blown', 'away', 'conference', 'want', 'thank', 'many', 'nice', 'comment', 'say', 'night', 'say', 'sincerely', 'partly', 'mock', 'sob', 'need', 'laughter', 'put', 'position', 'laughter', 'flew', 'air', 'force', 'two', 'eight', 'year', 'laughter', 'take', 'shoe', 'boot', 'get', 'airplane', 'laughter', 'applause', 'ill', 'tell', 'one', 'quick', 'story', 'illustrate', 'thats', 'like', 'laughter', 'true', 'story', 'every', 'bit', 'true', 'soon', 'tipper', 'left', 'mock', 'sob', 'white', 'house', 'laughter', 'driving', 'home', 'nashville', 'little', 'farm', '50', 'mile', 'east', 'nashville', 'driving', 'laughter', 'know', 'sound', 'like', 'little', 'thing', 'laughter', 'looked', 'rearview', 'mirror', 'sudden', 'hit', 'motorcade', 'back', 'laughter', 'youve', 'heard', 'phantom', 'limb', 'pain', 'laughter']


### Preprocessing to list of sentences

In [45]:
from nltk.tokenize import sent_tokenize
sentences = sent_tokenize(data)
processed_sentences = [preprocessing(s) for s in sentences if s.strip()]

print("Number of sentences:", len(processed_sentences))
print("First processed_sentences:", processed_sentences[0])

Number of sentences: 444404
First processed_sentences: ['thank', 'much', 'chris']


## **Build models**

### 1-gram model (Unigram model)

In [46]:
class UnigramModel:
  def __init__(self, tokens):
    self.unigrams = Counter(tokens)
    self.N = sum(self.unigrams.values())
    self.V = len(self.unigrams)

  def get_unigram_prob(self, wi):
    return (self.unigrams[wi] + 1) / (self.N + self.V)

  def sentence_prob(self, sentence_tokens):
    sent = sentence_tokens + ["</s>"]
    log_prob = 0.0
    for wi in sent:
      log_prob += math.log(self.get_unigram_prob(wi))
    return math.exp(log_prob)

  def perplexity(self, sentence_tokens):
    sent = sentence_tokens + ["</s>"]
    N = len(sent)
    log_prob = 0.0
    for wi in sent:
      log_prob += math.log(self.get_unigram_prob(wi))
    return math.exp(-log_prob / N)

### 2-gram model (Bigram model)

In [47]:
class BigramModel:
    def __init__(self, processed_sentences):
        self.unigrams = Counter()
        self.bigrams = Counter()
        for sent in processed_sentences:
            sent2 = ["<s>"] + sent + ["</s>"]
            self.unigrams.update(sent2)
            self.bigrams.update(zip(sent2[:-1], sent2[1:]))
        self.V = len(self.unigrams)

    def get_bigram_prob(self, wi, wi_1):
        return (self.bigrams[(wi_1, wi)] + 1) / (self.unigrams[wi_1] + self.V)

    def sentence_prob(self, sentence_tokens):
        sent2 = ["<s>"] + sentence_tokens + ["</s>"]
        log_prob = 0.0
        for i in range(1, len(sent2)):
            log_prob += math.log(self.get_bigram_prob(sent2[i], sent2[i-1]))
        return math.exp(log_prob)

    def perplexity(self, sentence_tokens):
      sent2 = ["<s>"] + sentence_tokens + ["</s>"]
      N = len(sent2) - 1
      log_prob = 0.0
      for i in range(1, len(sent2)):
        log_prob += math.log(self.get_bigram_prob(sent2[i], sent2[i-1]))
      return math.exp(-log_prob / N)

### 3-gram model (Trigram model)

In [48]:
class TrigramModel:
    def __init__(self, processed_sentences):
        self.unigrams = Counter()
        self.bigrams = Counter()
        self.trigrams = Counter()
        for sent in processed_sentences:
            sent3 = ["<s>", "<s>"] + sent + ["</s>"]
            self.unigrams.update(sent3)
            self.bigrams.update(zip(sent3[:-1], sent3[1:]))
            self.trigrams.update(zip(sent3[:-2], sent3[1:-1], sent3[2:]))
        self.V = len(self.unigrams)

    def get_trigram_prob(self, wi, wi_1, wi_2):
        return (self.trigrams[(wi_2, wi_1, wi)] + 1) / (self.bigrams[(wi_2, wi_1)] + self.V)

    def sentence_prob(self, sentence_tokens):
        sent3 = ["<s>", "<s>"] + sentence_tokens + ["</s>"]
        log_prob = 0.0
        for i in range(2, len(sent3)):
            log_prob += math.log(self.get_trigram_prob(sent3[i], sent3[i-1], sent3[i-2]))
        return math.exp(log_prob)

    def perplexity(self, sentence_tokens):
        sent3 = ["<s>", "<s>"] + sentence_tokens + ["</s>"]
        N = len(sent3) - 2
        log_prob = 0.0
        for i in range(2, len(sent3)):
            log_prob += math.log(self.get_trigram_prob(sent3[i], sent3[i-1], sent3[i-2]))
        return math.exp(-log_prob / N)

## Run the models

### **b.** Calculate the probability of a sentence and compute the Perplexity of a sentence based on 1-gram, 2-gram, and 3-gram models.

In [49]:
uni_model = UnigramModel(tokens)
bi_model = BigramModel(processed_sentences)
tri_model = TrigramModel(processed_sentences)

test1 = preprocessing("I love natural language processing")
test2 = preprocessing("Natural language processing I love")
def print_results(name, prob, ppl):
    print(f"{name:<10} | Prob: {prob:.2e} | Perplexity: {ppl:,.2f}")

print_results("Unigram prob:", uni_model.sentence_prob(test1), uni_model.perplexity(test1))
print_results("Bigram prob:", bi_model.sentence_prob(test1), bi_model.perplexity(test1))
print_results("Trigram prob:", tri_model.sentence_prob(test1), tri_model.perplexity(test1))

print("\n-- Wrong order --")
print_results("Unigram prob:", uni_model.sentence_prob(test2), uni_model.perplexity(test2))
print_results("Bigram prob:", bi_model.sentence_prob(test2), bi_model.perplexity(test2))
print_results("Trigram prob:", tri_model.sentence_prob(test2), tri_model.perplexity(test2))


Unigram prob: | Prob: 4.50e-21 | Perplexity: 11,732.28
Bigram prob: | Prob: 5.50e-19 | Perplexity: 4,486.04
Trigram prob: | Prob: 2.17e-22 | Perplexity: 21,513.29

-- Wrong order --
Unigram prob: | Prob: 4.50e-21 | Perplexity: 11,732.28
Bigram prob: | Prob: 4.55e-19 | Perplexity: 4,659.73
Trigram prob: | Prob: 3.67e-23 | Perplexity: 30,692.19


### **c.** Analyze the results (Provide your own examples of spelling errors and calculate the probability of two similar sentences, where one has the correct word order and the other has an incorrect word order).

In [50]:
# Compact automated sentence analysis
def analyze_pair(correct, wrong, desc, models):
    print(f"\nTest {models['counter']}: {desc}")

    test_correct = preprocessing(correct)
    test_wrong = preprocessing(wrong)

    uni_prob_correct = models['unigram'].sentence_prob(test_correct)
    uni_ppl_correct = models['unigram'].perplexity(test_correct)
    bi_prob_correct = models['bigram'].sentence_prob(test_correct)
    bi_ppl_correct = models['bigram'].perplexity(test_correct)
    tri_prob_correct = models['trigram'].sentence_prob(test_correct)
    tri_ppl_correct = models['trigram'].perplexity(test_correct)

    uni_prob_wrong = models['unigram'].sentence_prob(test_wrong)
    uni_ppl_wrong = models['unigram'].perplexity(test_wrong)
    bi_prob_wrong = models['bigram'].sentence_prob(test_wrong)
    bi_ppl_wrong = models['bigram'].perplexity(test_wrong)
    tri_prob_wrong = models['trigram'].sentence_prob(test_wrong)
    tri_ppl_wrong = models['trigram'].perplexity(test_wrong)

    probs = {
        'uni': [uni_prob_correct, uni_prob_wrong],
        'bi': [bi_prob_correct, bi_prob_wrong],
        'tri': [tri_prob_correct, tri_prob_wrong]
    }

    print(f"\nCorrect sentence: '{correct}'")
    print_results("Unigram", uni_prob_correct, uni_ppl_correct)
    print_results("Bigram", bi_prob_correct, bi_ppl_correct)
    print_results("Trigram", tri_prob_correct, tri_ppl_correct)

    print(f"\nWrong sentence: '{wrong}'")
    print_results("Unigram", uni_prob_wrong, uni_ppl_wrong)
    print_results("Bigram", bi_prob_wrong, bi_ppl_wrong)
    print_results("Trigram", tri_prob_wrong, tri_ppl_wrong)

    # Check which models correctly identify the right sentence
    results = []
    for name, prob_pair in probs.items():
        if prob_pair[0] > prob_pair[1]:
            ratio = prob_pair[0] / prob_pair[1] if prob_pair[1] > 0 else float('inf')
            results.append(f"{name.upper()}:Can({ratio:.1f}x)")
        else:
            results.append(f"{name.upper()}:Cannot")
    print(" | ".join(results))

    # Store results
    models['results'].append({
        'desc': desc,
        'uni_ok': probs['uni'][0] > probs['uni'][1],
        'bi_ok': probs['bi'][0] > probs['bi'][1],
        'tri_ok': probs['tri'][0] > probs['tri'][1]
    })
    models['counter'] += 1

# Test cases
tests = [
    ("Machine learning is very useful", "Useful very is learning machine", "Word Order"),
    ("Programming languages are important", "Programing languges are importent", "Spelling"),
    ("I enjoy studying computer science", "I enjoys study computer sciences", "Grammar"),
    ("Deep learning models require large datasets", "Deep learning require large datasets", "Missing Word"),
    ("Neural networks can learn patterns", "Neural networks can can learn learn patterns", "Extra Words"),
    ("Python is a programming language", "Python is a dangerous animal", "Semantic Error"),
    ("Data science involves statistics", "Statistics involves data science", "Subject Swap"),
    ("AI will change the world", "AI will changed the world", "Verb Tense")
]

# Initialize
models_dict = {
    'unigram': uni_model, 'bigram': bi_model, 'trigram': tri_model,
    'counter': 1, 'results': []
}

# Run tests
for correct, wrong, desc in tests:
    analyze_pair(correct, wrong, desc, models_dict)

# Summary
results = models_dict['results']
total = len(results)
uni_score = sum(r['uni_ok'] for r in results)
bi_score = sum(r['bi_ok'] for r in results)
tri_score = sum(r['tri_ok'] for r in results)

print("\n__Summary results:___\n")
print(f"Unigram: {uni_score}/{total} ({uni_score/total*100:.0f}%)")
print(f"Bigram:  {bi_score}/{total} ({bi_score/total*100:.0f}%)")
print(f"Trigram: {tri_score}/{total} ({tri_score/total*100:.0f}%)")

best = max(['Unigram', 'Bigram', 'Trigram'], key=lambda x: [uni_score, bi_score, tri_score][['Unigram', 'Bigram', 'Trigram'].index(x)])
print(f"{best}")


Test 1: Word Order

Correct sentence: 'Machine learning is very useful'
Unigram    | Prob: 8.82e-18 | Perplexity: 18,352.38
Bigram     | Prob: 2.75e-14 | Perplexity: 2,454.89
Trigram    | Prob: 2.61e-17 | Perplexity: 13,993.29

Wrong sentence: 'Useful very is learning machine'
Unigram    | Prob: 8.82e-18 | Perplexity: 18,352.38
Bigram     | Prob: 5.88e-16 | Perplexity: 6,421.64
Trigram    | Prob: 2.47e-19 | Perplexity: 44,868.91
UNI:Cannot | BI:Can(46.8x) | TRI:Can(105.7x)

Test 2: Spelling

Correct sentence: 'Programming languages are important'
Unigram    | Prob: 8.63e-18 | Perplexity: 18,450.52
Bigram     | Prob: 1.91e-15 | Perplexity: 4,783.63
Trigram    | Prob: 2.96e-19 | Perplexity: 42,869.39

Wrong sentence: 'Programing languges are importent'
Unigram    | Prob: 2.38e-26 | Perplexity: 2,546,162.83
Bigram     | Prob: 4.94e-21 | Perplexity: 119,302.48
Trigram    | Prob: 4.94e-21 | Perplexity: 119,301.25
UNI:Can(362669120.0x) | BI:Can(386870.8x) | TRI:Can(60.0x)

Test 3: Grammar

